In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType,
    TimestampType,
    FloatType,
)
import pyspark.sql.functions as F
from  pyspark.sql import Row
catalog_name = 'ecommerce'

In [0]:
df_product=spark.read.table(f'{catalog_name}.silver.slv_product')
df_category=spark.read.table(f'{catalog_name}.silver.slv_category')
df_brands=spark.read.table(f'{catalog_name}.silver.slv_brands')

df_product.show()

In [0]:
df_category.show()

In [0]:
df_brands.show()


In [0]:
df_product.createOrReplaceTempView("view_product")
df_category.createOrReplaceTempView("view_category")
df_brands.createOrReplaceTempView("view_brand")

In [0]:
%sql
select * from view_product;

In [0]:
%sql
use catalog ecommerce;

In [0]:
%sql
    select b.brand_code,b.brand_name,c.category_code,c.category_name from view_brand b join view_category c on b.category_code = c.category_code

In [0]:
%sql
create or replace table gold.gld_product as
with brand_categories as ( --cte 
  select
    b.brand_code,
    b.brand_name,
    c.category_code,
    c.category_name
  from
    view_brand b
      join view_category c
        on b.category_code = c.category_code
)
select
  p.product_id,
  p.sku,
  p.category_code,
  coalesce(bc.category_name, "Not Available") as category_name,
  p.brand_code,
  coalesce(bc.brand_name, "Not Available") as brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p.file_name,
  p.ingest_timestamp

from
  view_product p
    left join brand_categories bc
      on p.brand_code = bc.brand_code

In [0]:
x=df_product.join(df_brands,how="inner",on="brand_code").join(df_category,how="inner",on="category_code")
x.show()
# x=x.join(df_category,how="inner",on="category_code")
# x.show()

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}
country_state_map={
    "India":india_region,
    "Austrailia":australia_region,
    "United Kingdom":uk_region,
    "United States":us_region,
    "UAE":uae_region,
    "Singapore":singapore_region,
    "Canada":canada_region    
}

In [0]:
country_state_map["India"]

In [0]:
row=[]
for country, state in  country_state_map.items():
    for state,region in country_state_map[country].items():
        row.append(Row(Country=country, State=state, Region=region))
    #     print(f'{state},{region}')
    # print(f'{country} with the states {state}')
    # print('\n')

In [0]:
print(row)

In [0]:
df_country_state_map=spark.createDataFrame(row)
df_country_state_map.show()
# df_country_state_map.createOrReplaceTempView("view_country_state_map")
# # x=x.join(df_country_state_map,how="inner",on="Country")
# # x.show(

In [0]:
df_silver_customer=spark.read.table(f'{catalog_name}.silver.slv_customer')
df_silver_customer.show()
# df_silver_customer.createOrReplaceTempView("view_silver_customer")
# df_silver_customer=spark.sql("select * from view_silver_customer")

In [0]:
df_gold_customer=df_silver_customer.join(df_country_state_map,on=['Country',"State"],how="left")
df_gold_customer= df_gold_customer.fillna({'region': 'Other'})

df_gold_customer.show(truncate=False)

In [0]:
df_gold_customer.write.format("delta").mode("overwrite").option('mergeSchema',True).saveAsTable(f'{catalog_name}.gold.gld_customer')

### ****

In [0]:
df_silver_calender=spark.read.table(f'{catalog_name}.silver.slv_calendar')
df_silver_calender.show()

In [0]:
df_gold_calender=df_silver_calender.withColumn('date_id',F.date_format(F.col("date"),'yyyyMMdd').cast("int"))
df_gold_calender=df_gold_calender.withColumn('Month_name',F.date_format(F.col("date"),"MMMM"))

df_gold_calender = df_gold_calender.withColumn(
    "is_weekend",
    F.when(F.col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)
df_gold_calender.show()



In [0]:
desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "_ingested_at", "_source_file"]

df_gold_calender = df_gold_calender.select(desired_columns_order)

df_gold_calender.show()



In [0]:
df_gold_calender.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(
    f"{catalog_name}.gold.gld_calendar"
)

In [0]:
%sql
DESCRIBE EXTENDED ecommerce.gold.gld_calendar;